# Pregunta 1 — Clasificación Multietiqueta de Anomalías de Rodilla en Resonancia Magnética (CNN + Aprendizaje Débilmente Supervisado)

**Grupo 2 · Examen Parcial · Redes Neuronales y Aprendizaje Profundo · UNI FIIS 2026-II**

Este notebook ejecuta todo el proyecto con un solo comando (`run_all.sh`). Descarga el código del repositorio [pomazon/Pregunta1_Parcial_RNAP](https://github.com/pomazon/Pregunta1_Parcial_RNAP) en la versión entregada (`entrega-final`) y corre los diez pasos en orden: exploración, traducción de reportes, extracción de etiquetas, validación del extractor, selección de estudios, preprocesamiento de imágenes, los seis entrenamientos (A–F), Grad-CAM y figuras.

## Antes de ejecutar: configuración en Kaggle

1. **Aceptar las reglas de la competencia** *RSNA Knee Abnormality Detection* con tu cuenta (pestaña *Rules*). Sin eso, Kaggle no deja montar los datos.
2. **Añadir los datos:** panel derecho → *Input* → **+ Add Input** → buscar *RSNA Knee Abnormality Detection* → *Add*. Quedan en `/kaggle/input/competitions/rsna-knee-abnormality-detection`; la celda 1 también busca otras rutas.
3. **Settings → Accelerator: GPU T4 x1.** La traducción y el entrenamiento necesitan GPU.
4. **Settings → Internet: On.** Hace falta para clonar el repositorio, instalar dependencias y descargar los pesos abiertos de NLLB-200 y ResNet-50. Kaggle solo permite activar Internet si la cuenta tiene el teléfono verificado.
5. **Elegir el modo** en la celda 2: `"rapido"` (unos 25 minutos, para comprobar que todo corre) o `"completo"` (varias horas, reproduce los resultados del informe).

Luego, **Run All**. Para la corrida completa conviene usar **Save Version → Save & Run All (Commit)**: así corre en segundo plano aunque cierres el navegador, con un límite de 12 horas.

> Los datos no salen de Kaggle: los reportes se traducen con un modelo abierto que corre aquí mismo y no se envían a ningún servicio externo. Los resultados quedan en `/kaggle/working/Pregunta1_Parcial_RNAP/outputs/`. No subas esos resultados ni este notebook con sus salidas a GitHub, porque contienen datos derivados de la competencia.

## 1. Comprobar el entorno

In [ ]:
import os, glob, shutil, subprocess

# GPU
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No hay GPU. Activa Settings → Accelerator → GPU T4 x1 y vuelve a ejecutar."
print("GPU:", gpu.stdout.strip())

# Datos de la competencia
candidatos = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]
candidatos += [os.path.dirname(p) for p in glob.glob("/kaggle/input/*/train_series.csv")]
candidatos += [os.path.dirname(p) for p in glob.glob("/kaggle/input/*/*/train_series.csv")]
DATA_DIR = next((d for d in candidatos if os.path.exists(os.path.join(d, "train_series.csv"))), None)
assert DATA_DIR, "No encuentro los datos. Añade la competencia en Input → + Add Input."
print("Datos:", DATA_DIR)

# Internet
net = subprocess.run(["git", "ls-remote", "https://github.com/pomazon/Pregunta1_Parcial_RNAP.git"],
                     capture_output=True, text=True)
assert net.returncode == 0, "Sin Internet. Activa Settings → Internet → On."
print("Internet: OK")
print("Disco libre en /kaggle/working:", shutil.disk_usage("/kaggle/working").free // 2**30, "GB")

## 2. Elegir el modo de ejecución

In [ ]:
# "rapido":   prueba de extremo a extremo con pocos estudios y 1 época (~25 min).
#             Comprueba que todo corre; sus números NO son los resultados del informe.
# "completo": reproduce los resultados del informe (varias horas en una GPU T4).
MODO = "rapido"

RAMA = "entrega-final"   # versión entregada del código
REPO = "https://github.com/pomazon/Pregunta1_Parcial_RNAP.git"
assert MODO in ("rapido", "completo"), "MODO debe ser rapido o completo"


## 3. Descargar el código

In [ ]:
%cd /kaggle/working
if os.path.isdir("Pregunta1_Parcial_RNAP"):
    shutil.rmtree("Pregunta1_Parcial_RNAP")   # empezar siempre desde una copia limpia
!git clone --quiet --branch {RAMA} {REPO}
%cd /kaggle/working/Pregunta1_Parcial_RNAP
!git log -1 --format="Versión del código: %h (%cd)" --date=short

## 4. Ejecutar el proyecto (comando único)

Esta celda instala `requirements.txt` y corre los pasos 1 a 10 de `run_all.sh`. Cada paso imprime su avance y al final indica cuántos minutos tomó.

In [ ]:
FLAG = "--quick" if MODO == "rapido" else ""
!bash run_all.sh {FLAG} "{DATA_DIR}"

## 5. Resumen de resultados

In [ ]:
import pandas as pd
pd.set_option("display.width", 160)

res = "outputs/runs/summary.csv"
if os.path.exists(res):
    print("AUC macro y F1 de cada modelo en los estudios de prueba:")
    display(pd.read_csv(res))
else:
    print("No se encontró", res, "- revisa los mensajes de la celda 4.")

if MODO == "rapido":
    print("\nModo rápido: estos números solo confirman que el código corre; no son los resultados del informe.")

print("\nCarpetas generadas en outputs/:")
for d in sorted(os.listdir("outputs")):
    ruta = os.path.join("outputs", d)
    if os.path.isdir(ruta):
        print(f"  {d}/  ({len(os.listdir(ruta))} elementos)")

## Si una sola sesión no alcanza

La corrida completa puede pasar del límite de 12 horas de Kaggle. En ese caso se reparte en tramos, cada uno en un notebook nuevo que añade como *Input* la salida del anterior (*+ Add Input → Your Work → la versión guardada del notebook anterior*). Los scripts buscan en `/kaggle/input` lo que no encuentran en `outputs/`, así que no hay que mover archivos.

| Tramo | Qué correr después de las celdas 1 a 3 | Input adicional |
|---|---|---|
| 1 | `!pip install -q -r requirements.txt` y los scripts `01` a `06` (como en `run_all.sh`) | — |
| 2 | `!pip install -q -r requirements.txt` y `!bash scripts/run_experiments.sh` | Salida del tramo 1 |
| 3 | `!pip install -q -r requirements.txt`, las tres llamadas a `scripts/09_gradcam.py` y `!python scripts/10_figures.py` | Salidas de los tramos 1 y 2 |